# 03 — Creación de nuevas métricas

Partimos del dataset limpio `Matriz_V1.xlsx` (5 085 filas × 28 columnas) generado en el notebook anterior.  
El objetivo es crear **métricas derivadas**: una métrica agregada de golpeos potentes y la **normalización por Phase Duration (min)** de las 4 VD seleccionadas, para obtener tasas por minuto comparables entre fases de distinta duración.

## 1. Carga del dataset limpio

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("../Datos/Matriz_V1.xlsx")
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nColumnas:\n{list(df.columns)}")
df.head()

Dimensiones: 4621 filas × 28 columnas

Columnas:
['Session Id', 'Phase Id', 'Phase Duration (min)', 'Participated Players', 'Position Category', 'Position', 'Total Touches (#)', 'Left Leg Touches (#)', 'Right Leg Touches (#)', 'Releases (#)', 'RV Zone 1 [0-5( m/s)]', 'RV Zone 2 [5-10( m/s)]', 'RV Zone 3 [10-15( m/s)]', 'RV Zone 4 [15-20( m/s)]', 'RV Zone 5 [20-25( m/s)]', 'RV Zone 6 [> 25( m/s)]', 'Distance Covered (m)', 'HID Covered (m) Zone 1 [> 4(m/s)]', 'SD Covered (m) [> 5.5(m/s)]', 'Release Velocity Max (m/s)', 'Release Velocity Avg (m/s)', 'Release Index Beta - Total', 'Player Id', 'Agrupacion', 'Espacio', 'Polaridad', 'Equilibrio', 'NombreCorrecto']


,Session Id,Phase Id,Phase Duration (min),Participated Players,Position Category,Position,Total Touches (#),Left Leg Touches (#),Right Leg Touches (#),Releases (#),...,SD Covered (m) [> 5.5(m/s)],Release Velocity Max (m/s),Release Velocity Avg (m/s),Release Index Beta - Total,Player Id,Agrupacion,Espacio,Polaridad,Equilibrio,NombreCorrecto
0,1656564,1656692,13,22,Midfielders,Defensive Midfielder,33,11,22,10,...,2,18.4,11.8,11.8,111224,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor
1,1656564,1656692,13,22,Forwards,Right Winger,21,9,12,5,...,6,16.6,13.2,6.6,113152,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor
2,1656564,1656692,13,22,Defenders,Right Center Back,34,14,20,13,...,5,16.2,10.9,14.2,113153,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor
3,1656564,1656692,13,22,Forwards,Right Winger,3,1,2,1,...,0,5.7,5.7,0.6,113155,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor
4,1656564,1656692,13,22,Midfielders,Center Midfielder,18,4,14,5,...,55,20.2,15.5,7.8,113159,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor


## 2. Creación de nuevas métricas

### 2.1 `Golpeos +15 m/s`

Suma de las zonas de velocidad de golpeo superiores a 15 m/s:
- `RV Zone 4 [15-20( m/s)]`
- `RV Zone 5 [20-25( m/s)]`
- `RV Zone 6 [> 25( m/s)]`

### 2.2 Renombrado de SD Covered → High Intensity Distance (20 km/h)

La variable `SD Covered (m) [> 5.5(m/s)]` (distancia recorrida a > 5.5 m/s ≈ 19.8 km/h) se renombra a **High Intensity Distance (20 km/h)** para reflejar mejor su significado deportivo: distancia a alta intensidad según el umbral más alineado con la literatura científica.

### 2.3 Métricas normalizadas por minuto

Dividimos las 4 VD seleccionadas por `Phase Duration (min)` para obtener **tasas por minuto**, eliminando el efecto de la duración de la fase:

| Métrica original | Métrica normalizada |
|:---|:---|
| Total Touches (#) | `Total Touches / min` |
| Golpeos +15 m/s | `Golpeos +15 m/s / min` |
| Distance Covered (m) | `Distance Covered (m) / min` |
| High Intensity Distance (20 km/h) | `High Intensity Distance (20 km/h) / min` |

In [2]:
# --- 2.1 Golpeos +15 m/s ---
df["Golpeos +15 m/s"] = (
    df["RV Zone 4 [15-20( m/s)]"]
    + df["RV Zone 5 [20-25( m/s)]"]
    + df["RV Zone 6 [> 25( m/s)]"]
)

# --- 2.2 Renombrado de SD Covered → High Intensity Distance (20 km/h) ---
df.rename(columns={
    "SD Covered (m) [> 5.5(m/s)]": "High Intensity Distance (20 km/h)"
}, inplace=True)

# --- 2.3 Métricas normalizadas por minuto ---
duracion = df["Phase Duration (min)"]

df["Total Touches / min"]                      = df["Total Touches (#)"]                    / duracion
df["Golpeos +15 m/s / min"]                    = df["Golpeos +15 m/s"]                      / duracion
df["Distance Covered (m) / min"]               = df["Distance Covered (m)"]                 / duracion
df["High Intensity Distance (20 km/h) / min"]  = df["High Intensity Distance (20 km/h)"]    / duracion

# Resumen de las nuevas columnas
nuevas_cols = [
    "Golpeos +15 m/s",
    "High Intensity Distance (20 km/h)",
    "Total Touches / min",
    "Golpeos +15 m/s / min",
    "Distance Covered (m) / min",
    "High Intensity Distance (20 km/h) / min",
]

print(f"✅ Creadas/renombradas {len(nuevas_cols)} columnas:")
for col in nuevas_cols:
    print(f"   • {col}")

print(f"\nDimensiones actualizadas: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nEstadísticas de las nuevas métricas:")
df[nuevas_cols].describe().round(2)

✅ Creadas/renombradas 6 columnas:
   • Golpeos +15 m/s
   • High Intensity Distance (20 km/h)
   • Total Touches / min
   • Golpeos +15 m/s / min
   • Distance Covered (m) / min
   • High Intensity Distance (20 km/h) / min

Dimensiones actualizadas: 4621 filas × 33 columnas

Estadísticas de las nuevas métricas:


,Golpeos +15 m/s,High Intensity Distance (20 km/h),Total Touches / min,Golpeos +15 m/s / min,Distance Covered (m) / min,High Intensity Distance (20 km/h) / min
count,4621.00,4621.00,4621.00,4621.00,4621.00,4621.00
mean,2.32,29.07,2.40,0.16,77.27,1.94
std,2.99,49.31,1.80,0.22,30.25,3.89
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,1.17,0.00,55.82,0.00
50%,1.00,7.00,2.00,0.10,77.08,0.50
75%,3.00,36.00,3.18,0.22,99.25,2.38
max,38.00,483.00,14.42,4.50,172.00,67.00


## 3. Eliminación de columnas redundantes

Tras seleccionar las 4 VD finales en el notebook anterior y crear las métricas derivadas, eliminamos las **13 columnas** que ya no son necesarias:

- **Left / Right Leg Touches**: subconjuntos de Total Touches.
- **Releases (#)**: redundante con Total Touches (r ≈ 0.88).
- **RV Zones 1–6**: capturadas por Golpeos +15 m/s (las de alta velocidad) o redundantes con Releases (las de baja velocidad).
- **HID Covered**: descartada en favor de SD Covered (renombrada a High Intensity Distance). Su umbral de 4 m/s (≈ 14.4 km/h) es demasiado bajo para alta intensidad real y presenta mayor correlación con Distance Covered (r ≈ 0.81).
- **Release Velocity Max / Avg**: métricas de velocidad puntual, descartadas.
- **Release Index Beta — Total**: redundante con Total Touches y Releases (r ≈ 0.96).

In [3]:
# --- 3. Eliminación de columnas redundantes ---
cols_eliminar = [
    "Session Id",
    "Participated Players",
    "Left Leg Touches (#)",
    "Right Leg Touches (#)",
    "Releases (#)",
    "RV Zone 1 [0-5( m/s)]",
    "RV Zone 2 [5-10( m/s)]",
    "RV Zone 3 [10-15( m/s)]",
    "RV Zone 4 [15-20( m/s)]",
    "RV Zone 5 [20-25( m/s)]",
    "RV Zone 6 [> 25( m/s)]",
    "HID Covered (m) Zone 1 [> 4(m/s)]",
    "Release Velocity Max (m/s)",
    "Release Velocity Avg (m/s)",
    "Release Index Beta - Total",
]

print(f"Columnas antes: {df.shape[1]}")
df.drop(columns=cols_eliminar, inplace=True)
print(f"Columnas después: {df.shape[1]}")
print(f"\n🗑️ Eliminadas {len(cols_eliminar)} columnas:")
for col in cols_eliminar:
    print(f"   • {col}")

print(f"\nColumnas restantes ({df.shape[1]}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

Columnas antes: 33
Columnas después: 18

🗑️ Eliminadas 15 columnas:
   • Session Id
   • Participated Players
   • Left Leg Touches (#)
   • Right Leg Touches (#)
   • Releases (#)
   • RV Zone 1 [0-5( m/s)]
   • RV Zone 2 [5-10( m/s)]
   • RV Zone 3 [10-15( m/s)]
   • RV Zone 4 [15-20( m/s)]
   • RV Zone 5 [20-25( m/s)]
   • RV Zone 6 [> 25( m/s)]
   • HID Covered (m) Zone 1 [> 4(m/s)]
   • Release Velocity Max (m/s)
   • Release Velocity Avg (m/s)
   • Release Index Beta - Total

Columnas restantes (18):
   1. Phase Id
   2. Phase Duration (min)
   3. Position Category
   4. Position
   5. Total Touches (#)
   6. Distance Covered (m)
   7. High Intensity Distance (20 km/h)
   8. Player Id
   9. Agrupacion
  10. Espacio
  11. Polaridad
  12. Equilibrio
  13. NombreCorrecto
  14. Golpeos +15 m/s
  15. Total Touches / min
  16. Golpeos +15 m/s / min
  17. Distance Covered (m) / min
  18. High Intensity Distance (20 km/h) / min


## 4. Columna "Equipo" — Mapeo de NombreCorrecto a categoría de equipo

Se crea una nueva columna `Equipo` que agrupa los valores de `NombreCorrecto` (nombre específico de cada equipo/categoría) en **5 categorías** de equipo:

| Equipo | NombreCorrecto incluidos |
|:---|:---|
| Neskak | Neskak A, Neskak B, Neskak C |
| Infantil | Infantil Txiki, Infantil Handi |
| Cadete | Cadete Txiki, Cadete Vasca |
| Juvenil | EASO, Juvenil División de Honor |
| Senior Masculino | Real Sociedad C, SANSE, Real Sociedad |

In [4]:
# --- 5. Creación de columna "Equipo" (mapeo de NombreCorrecto) ---
MAPEO_EQUIPO = {
    "Neskak A":                   "Neskak",
    "Neskak B":                   "Neskak",
    "Neskak C":                   "Neskak",
    "Infantil Txiki":             "Infantil",
    "Infantil Handi":             "Infantil",
    "Cadete Txiki":               "Cadete",
    "Cadete Vasca":               "Cadete",
    "EASO":                       "Juvenil",
    "Juvenil División de Honor":  "Juvenil",
    "Real Sociedad C":            "Senior Masculino",
    "SANSE":                      "Senior Masculino",
    "Real Sociedad":              "Senior Masculino",
}

df["GrupoEdad"] = df["NombreCorrecto"].map(MAPEO_EQUIPO)

# Verificación: ¿algún NombreCorrecto sin mapear?
sin_mapear = df["GrupoEdad"].isna().sum()
if sin_mapear > 0:
    nombres_sin = df.loc[df["GrupoEdad"].isna(), "NombreCorrecto"].unique()
    print(f"⚠️  {sin_mapear} filas sin mapear: {nombres_sin.tolist()}")
else:
    print("✅ Columna 'GrupoEdad' creada — todos los valores mapeados correctamente")

# Resumen del mapeo
print(f"\nMapeo NombreCorrecto → GrupoEdad:")
for equipo in sorted(df["GrupoEdad"].unique()):
    nombres = sorted(df.loc[df["GrupoEdad"] == equipo, "NombreCorrecto"].unique())
    print(f"   {equipo}: {', '.join(nombres)}")

print(f"\nDistribución:")
print(df["GrupoEdad"].value_counts().to_string())

✅ Columna 'GrupoEdad' creada — todos los valores mapeados correctamente

Mapeo NombreCorrecto → GrupoEdad:
   Cadete: Cadete Txiki, Cadete Vasca
   Infantil: Infantil Handi, Infantil Txiki
   Juvenil: EASO, Juvenil División de Honor
   Neskak: Neskak A, Neskak B, Neskak C
   Senior Masculino: Real Sociedad, Real Sociedad C, SANSE

Distribución:
GrupoEdad
Neskak              1113
Senior Masculino    1096
Juvenil             1042
Cadete               734
Infantil             636


## 5. Reordenación de columnas

Organizamos las columnas en un orden lógico:
1. **Identificación y contexto**: NombreCorrecto, GrupoEdad, Phase Id, Agrupación, Espacio, Polaridad, Equilibrio, Phase Duration, Position Category, Position, Player Id.
2. **Métricas de rendimiento** (intercalando total y por minuto): Total Touches → Total Touches/min → Golpeos +15 → Golpeos +15/min → Distance → Distance/min → High Intensity Distance → High Intensity Distance/min.

In [5]:
# --- 5. Reordenación de columnas ---
orden_cols = [
    # Identificación y contexto
    "Phase Id",
    "Agrupacion",
    "Espacio",
    "Polaridad",
    "Equilibrio",
    "NombreCorrecto",
    "GrupoEdad",
    "Phase Duration (min)",
    "Player Id",
    "Position Category",
    "Position",
    # Métricas: total + per min intercaladas
    "Total Touches (#)",
    "Total Touches / min",
    "Golpeos +15 m/s",
    "Golpeos +15 m/s / min",
    "Distance Covered (m)",
    "Distance Covered (m) / min",
    "High Intensity Distance (20 km/h)",
    "High Intensity Distance (20 km/h) / min",
]

df = df[orden_cols]

print(f"✅ Columnas reordenadas ({df.shape[1]}):\n")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

print(f"\n{df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

✅ Columnas reordenadas (19):

   1. Phase Id
   2. Agrupacion
   3. Espacio
   4. Polaridad
   5. Equilibrio
   6. NombreCorrecto
   7. GrupoEdad
   8. Phase Duration (min)
   9. Player Id
  10. Position Category
  11. Position
  12. Total Touches (#)
  13. Total Touches / min
  14. Golpeos +15 m/s
  15. Golpeos +15 m/s / min
  16. Distance Covered (m)
  17. Distance Covered (m) / min
  18. High Intensity Distance (20 km/h)
  19. High Intensity Distance (20 km/h) / min

4,621 filas × 19 columnas


,Phase Id,Agrupacion,Espacio,Polaridad,Equilibrio,NombreCorrecto,GrupoEdad,Phase Duration (min),Player Id,Position Category,Position,Total Touches (#),Total Touches / min,Golpeos +15 m/s,Golpeos +15 m/s / min,Distance Covered (m),Distance Covered (m) / min,High Intensity Distance (20 km/h),High Intensity Distance (20 km/h) / min
0,1656692,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor,Juvenil,13,111224,Midfielders,Defensive Midfielder,33,2.538462,3,0.230769,1101,84.692308,2,0.153846
1,1656692,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor,Juvenil,13,113152,Forwards,Right Winger,21,1.615385,3,0.230769,916,70.461538,6,0.461538
2,1656692,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor,Juvenil,13,113153,Defenders,Right Center Back,34,2.615385,2,0.153846,1094,84.153846,5,0.384615
3,1656692,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor,Juvenil,13,113155,Forwards,Right Winger,3,0.230769,0,0.000000,781,60.076923,0,0.000000
4,1656692,grande,reducido,Polarizado,Equilibrio,Juvenil División de Honor,Juvenil,13,113159,Midfielders,Center Midfielder,18,1.384615,3,0.230769,1479,113.769231,55,4.230769


In [6]:
# --- 5. Exportación a Matriz_V2.xlsx ---
ruta_salida = "../Datos/Matriz_V2.xlsx"
df.to_excel(ruta_salida, index=False)
print(f"✅ Dataset exportado → {ruta_salida}")
print(f"   {df.shape[0]:,} filas × {df.shape[1]} columnas")

✅ Dataset exportado → ../Datos/Matriz_V2.xlsx
   4,621 filas × 19 columnas


---

## Resumen del notebook

### Entrada
- `Matriz_V1.xlsx` — 5 085 filas × 28 columnas (dataset limpio del notebook 01).

### Transformaciones realizadas

| Paso | Descripción | Resultado |
|:---:|:---|:---|
| 1 | Creación de **Golpeos +15 m/s** (RV Z4 + Z5 + Z6) | +1 columna |
| 2 | Renombrado de `SD Covered (m) [> 5.5(m/s)]` → **High Intensity Distance (20 km/h)** | renombrada |
| 3 | Normalización de las 4 VD por **Phase Duration (min)** → Total Touches/min, Golpeos +15/min, Distance/min, High Intensity Distance/min | +4 columnas |
| 4 | Eliminación de **15 columnas** redundantes (Session Id, Participated Players, Left/Right Leg, Releases, RV Zones 1–6, HID Covered, RV Max/Avg, RI Beta) | −15 columnas |
| 5 | Creación de columna **GrupoEdad** (mapeo de NombreCorrecto → categoría de equipo) | +1 columna |
| 6 | Reordenación lógica: contexto → métricas (total + /min intercaladas) | — |

### Mapeo NombreCorrecto → GrupoEdad

| GrupoEdad | NombreCorrecto |
|:---|:---|
| Neskak | Neskak A, Neskak B, Neskak C |
| Infantil | Infantil Txiki, Infantil Handi |
| Cadete | Cadete Txiki, Cadete Vasca |
| Juvenil | EASO, Juvenil División de Honor |
| Senior Masculino | Real Sociedad C, SANSE, Real Sociedad |

### Salida
- `Matriz_V2.xlsx` — **5 085 filas × 19 columnas**

| # | Columna | Tipo |
|:-:|:---|:---|
| 1 | Phase Id | Contexto |
| 2 | Agrupacion | Contexto |
| 3 | Espacio | Contexto |
| 4 | Polaridad | Contexto |
| 5 | Equilibrio | Contexto |
| 6 | NombreCorrecto | Contexto |
| 7 | GrupoEdad | Contexto |
| 8 | Phase Duration (min) | Contexto |
| 9 | Player Id | Contexto |
| 10 | Position Category | Contexto |
| 11 | Position | Contexto |
| 12 | Total Touches (#) | VD absoluta |
| 13 | Total Touches / min | VD normalizada |
| 14 | Golpeos +15 m/s | VD absoluta |
| 15 | Golpeos +15 m/s / min | VD normalizada |
| 16 | Distance Covered (m) | VD absoluta |
| 17 | Distance Covered (m) / min | VD normalizada |
| 18 | High Intensity Distance (20 km/h) | VD absoluta |
| 19 | High Intensity Distance (20 km/h) / min | VD normalizada |